# RAG Pipeline with RunPod Serverless

This notebook demonstrates a complete Retrieval-Augmented Generation (RAG) pipeline using:
- **RunPod Serverless LLM** for text generation
- **RunPod Infinity Embedding Worker** for document embeddings
- **FAISS** for vector storage and similarity search
- **LangChain** for orchestration
- **PDF Support** for loading documents from the `data/` folder

## What is RAG?
RAG combines document retrieval with language model generation to provide more accurate, context-aware responses by grounding the LLM's output in relevant source documents.

## How to Use:
1. Place your PDF files in the `data/` folder
2. Run all cells sequentially
3. Ask questions about your documents!

## 1. Install Required Dependencies

First, we need to install all the necessary packages for our RAG pipeline.

In [ ]:
# Install required packages
# Uncomment the line below if running for the first time
# !pip install langchain langchain-runpod langchain-community langchain-classic faiss-cpu requests pypdf

## 2. Import Libraries

Import all necessary libraries for building our RAG pipeline.

In [1]:
import os
import time
import requests
from typing import List
from pathlib import Path

# LangChain imports
from langchain_runpod import ChatRunPod  # RunPod LLM wrapper
from langchain_core.embeddings import Embeddings  # Base class for embeddings
from langchain_community.vectorstores import FAISS  # Vector database
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Text splitting
from langchain_classic.chains import RetrievalQA  # RAG chain
from langchain_core.documents import Document  # Document wrapper
from langchain_community.document_loaders import PyPDFLoader  # PDF loader

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 3. Create Custom RunPod Embeddings Class

Since there's no official LangChain integration for RunPod embeddings yet, we'll create a custom class that:
- Connects to the RunPod Infinity Embedding worker endpoint
- Handles the async job submission and polling
- Formats requests and responses according to the worker's requirements

In [2]:
class RunPodEmbeddings(Embeddings):
    """
    Custom embeddings class for RunPod Infinity Embedding worker.
    
    The RunPod worker uses OpenAI-compatible API format and requires:
    - Input format: {"input": {"model": "model_name", "input": "text"}}
    - Output format: {"data": [{"embedding": [...], "index": 0}], "model": "...", "usage": {...}}
    """
    
    def __init__(self, endpoint_id: str, api_key: str):
        """
        Initialize the RunPod embeddings client.
        
        Args:
            endpoint_id: Your RunPod serverless endpoint ID (e.g., "ykby1zol2xhrmo")
            api_key: Your RunPod API key
        """
        self.endpoint_id = endpoint_id
        self.api_key = api_key
        self.base_url = f"https://api.runpod.ai/v2/{endpoint_id}"

    def _get_embedding(self, text: str) -> List[float]:
        """
        Get embedding vector for a single text string.
        
        This method:
        1. Submits an async job to RunPod
        2. Polls the status endpoint until completion
        3. Extracts and returns the embedding vector
        
        Args:
            text: The text string to embed
            
        Returns:
            List of floats representing the embedding vector (384 dimensions for bge-small-en-v1.5)
        """
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.api_key}"
        }
        
        # RunPod Infinity Embedding worker expects this specific format
        payload = {
            "input": {
                "model": "BAAI/bge-small-en-v1.5",  # The embedding model to use
                "input": text  # Note: uses "input" not "prompt"
            }
        }

        # Step 1: Submit the embedding job to RunPod
        response = requests.post(f"{self.base_url}/run", json=payload, headers=headers)
        response.raise_for_status()
        job_id = response.json()['id']

        # Step 2: Poll for results until the job completes
        while True:
            status_response = requests.get(f"{self.base_url}/status/{job_id}", headers=headers)
            status_response.raise_for_status()
            data = status_response.json()

            if data['status'] == 'COMPLETED':
                # Step 3: Extract embedding from OpenAI-compatible response format
                output = data['output']
                
                # The response has structure: {"data": [{"embedding": [...], "index": 0}], ...}
                if isinstance(output, dict) and 'data' in output:
                    if len(output['data']) > 0 and 'embedding' in output['data'][0]:
                        return output['data'][0]['embedding']
                    else:
                        raise ValueError(f"Unexpected data format: {output}")
                else:
                    raise ValueError(f"Unexpected output format from embedding endpoint: {output}")
            elif data['status'] in ['FAILED', 'CANCELLED']:
                raise Exception(f"Embedding job failed: {data}")

            # Wait 1 second before checking again
            time.sleep(1)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """
        Embed a list of documents (used by FAISS for indexing).
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            List of embedding vectors, one for each input text
        """
        return [self._get_embedding(text) for text in texts]

    def embed_query(self, text: str) -> List[float]:
        """
        Embed a query string (used by FAISS for searching).
        
        Args:
            text: The query text to embed
            
        Returns:
            Embedding vector for the query
        """
        return self._get_embedding(text)

print("✅ RunPodEmbeddings class created successfully!")

✅ RunPodEmbeddings class created successfully!


## 4. Configuration

Set up your RunPod API credentials and endpoint IDs.

**Important:** Replace these with your actual RunPod credentials!

In [ ]:
# RunPod API credentials
RUNPOD_API_KEY = "XX"
os.environ["RUNPOD_API_KEY"] = RUNPOD_API_KEY

# RunPod Serverless Endpoint IDs
RUNPOD_LLM_ENDPOINT_ID = "XX"  # Your vLLM endpoint for text generation
RUNPOD_EMBEDDING_ENDPOINT_ID = "XX"  # Your Infinity Embedding endpoint

# Data and storage paths
DATA_FOLDER = "data"  # Folder containing PDF files
VECTOR_STORE_PATH = "vector_store"  # Path to save/load vector store

print("✅ Configuration set!")
print(f"   LLM Endpoint: {RUNPOD_LLM_ENDPOINT_ID}")
print(f"   Embedding Endpoint: {RUNPOD_EMBEDDING_ENDPOINT_ID}")
print(f"   Data Folder: {DATA_FOLDER}")
print(f"   Vector Store Path: {VECTOR_STORE_PATH}")

✅ Configuration set!
   LLM Endpoint: 9o63bm6yja2uoc
   Embedding Endpoint: rw56hi9o0tsxri
   Data Folder: data
   Vector Store Path: vector_store


## 5. Initialize Components

Now we'll initialize our LLM and embedding models.

In [4]:
# Initialize RunPod LLM
# ChatRunPod automatically handles the /runsync endpoint and request formatting
print("Initializing RunPod LLM...")
llm = ChatRunPod(
    endpoint_id=RUNPOD_LLM_ENDPOINT_ID,  # Just the ID, not the full URL
    model_kwargs={
        "temperature": 0.7,  # Controls randomness (0=deterministic, 1=creative)
        "max_tokens": 2000,  # Maximum length of generated response
    }
)
print("✅ RunPod LLM initialized!")

# Initialize RunPod embeddings using our custom class
print("\nInitializing RunPod embeddings (BAAI/bge-small-en-v1.5)...")
embeddings = RunPodEmbeddings(
    endpoint_id=RUNPOD_EMBEDDING_ENDPOINT_ID,
    api_key=RUNPOD_API_KEY
)
print("✅ RunPod embeddings initialized!")

Initializing RunPod LLM...
✅ RunPod LLM initialized!

Initializing RunPod embeddings (BAAI/bge-small-en-v1.5)...
✅ RunPod embeddings initialized!


## 6. Load PDF Documents from Data Folder

This section loads all PDF files from the `data/` folder and extracts their text.

**Instructions:**
1. Place your PDF files in the `data/` folder
2. Run this cell to load and process them

In [5]:
# Create data folder if it doesn't exist
os.makedirs(DATA_FOLDER, exist_ok=True)

# Find all PDF files in the data folder
data_path = Path(DATA_FOLDER)
pdf_files = list(data_path.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF file(s) in '{DATA_FOLDER}' folder:")
for pdf in pdf_files:
    print(f"  - {pdf.name}")

# Load documents from PDFs
documents = []

if len(pdf_files) > 0:
    print("\nLoading PDF documents...")
    
    for pdf_file in pdf_files:
        print(f"  Loading: {pdf_file.name}...")
        
        # PyPDFLoader loads PDF and extracts text from each page
        # It creates a Document object for each page with metadata
        loader = PyPDFLoader(str(pdf_file))
        pdf_docs = loader.load()
        
        # Add source filename to metadata
        for doc in pdf_docs:
            doc.metadata['filename'] = pdf_file.name
        
        documents.extend(pdf_docs)
        print(f"    ✅ Loaded {len(pdf_docs)} pages")
    
    print(f"\n✅ Total documents loaded: {len(documents)} pages from {len(pdf_files)} PDF(s)")
    
    # Show sample document
    if documents:
        print("\nSample document:")
        print(f"  Source: {documents[0].metadata.get('filename', 'unknown')}")
        print(f"  Page: {documents[0].metadata.get('page', 'unknown')}")
        print(f"  Content preview: {documents[0].page_content[:200]}...")
else:
    # If no PDFs found, use sample documents
    print("\n⚠️  No PDF files found. Using sample documents instead.")
    print("   To use your own PDFs, place them in the 'data/' folder and re-run this cell.\n")
    
    sample_texts = [
        "RunPod is a cloud computing platform that provides GPU instances for AI and machine learning workloads.",
        "Serverless GPUs on RunPod automatically scale based on demand and you only pay for actual usage.",
        "RAG (Retrieval Augmented Generation) combines document retrieval with LLM generation for better responses.",
        "LangChain is a framework for developing applications powered by language models.",
        "FAISS (Facebook AI Similarity Search) is a library for efficient similarity search of dense vectors.",
    ]
    
    documents = [
        Document(
            page_content=text,
            metadata={"source": f"sample_doc_{i}", "chunk_id": i}
        )
        for i, text in enumerate(sample_texts)
    ]
    print(f"✅ Created {len(documents)} sample documents")

Found 1 PDF file(s) in 'data' folder:
  - Understanding_Climate_Change.pdf

Loading PDF documents...
  Loading: Understanding_Climate_Change.pdf...
    ✅ Loaded 33 pages

✅ Total documents loaded: 33 pages from 1 PDF(s)

Sample document:
  Source: Understanding_Climate_Change.pdf
  Page: 0
  Content preview: Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the plane...


## 7. Split Documents into Chunks

For better embedding quality and retrieval precision, we split documents into smaller chunks.

**Why chunking matters:**
- Embeddings work best on focused, coherent text segments
- Smaller chunks allow more precise retrieval
- Ensures content fits within model context windows

In [6]:
print("Splitting documents into chunks...\n")

# RecursiveCharacterTextSplitter intelligently splits text while preserving structure
# It tries different separators in order to maintain logical boundaries
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,  # Maximum characters per chunk (larger for PDFs)
    chunk_overlap=150,  # Overlap between chunks to preserve context across boundaries
    separators=["\n\n", "\n", ". ", " ", ""],  # Try these separators in order of preference
    length_function=len,  # Function to measure text length
)

# Split documents while preserving all metadata (filename, page number, etc.)
split_docs = text_splitter.split_documents(documents)

print(f"✅ Split {len(documents)} document(s) into {len(split_docs)} chunks")
print(f"\nChunking statistics:")
print(f"  Original documents: {len(documents)}")
print(f"  Final chunks: {len(split_docs)}")
print(f"  Avg chunks per document: {len(split_docs) / len(documents):.1f}")

# Show sample chunk
if split_docs:
    print(f"\nSample chunk:")
    print(f"  Length: {len(split_docs[0].page_content)} characters")
    print(f"  Metadata: {split_docs[0].metadata}")
    print(f"  Content preview: {split_docs[0].page_content[:200]}...")

Splitting documents into chunks...

✅ Split 33 document(s) into 125 chunks

Chunking statistics:
  Original documents: 33
  Final chunks: 125
  Avg chunks per document: 3.8

Sample chunk:
  Length: 765 characters
  Metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2024-07-13T20:17:34+03:00', 'author': 'Nir', 'moddate': '2024-07-13T20:17:34+03:00', 'source': 'data/Understanding_Climate_Change.pdf', 'total_pages': 33, 'page': 0, 'page_label': '1', 'filename': 'Understanding_Climate_Change.pdf'}
  Content preview: Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the plane...


## 8. Create or Load Vector Store

This section either:
- **Creates** a new vector store by embedding all documents (takes time)
- **Loads** an existing vector store from disk (fast)

The vector store will be saved to the `vector_store/` folder for future use.

In [7]:
import os

# Check if vector store already exists
vector_store_exists = os.path.exists(VECTOR_STORE_PATH) and os.path.exists(f"{VECTOR_STORE_PATH}/index.faiss")

if vector_store_exists:
    print(f"📁 Found existing vector store at '{VECTOR_STORE_PATH}'")
    print("   Loading from disk (this is fast)...\n")
    
    # Load the pre-computed vector store from disk
    # This is much faster than re-embedding all documents
    vectorstore = FAISS.load_local(
        VECTOR_STORE_PATH,
        embeddings,
        allow_dangerous_deserialization=True  # Required for security (we trust our own data)
    )
    
    print(f"✅ Vector store loaded successfully!")
    print(f"   Ready to answer questions based on previously embedded documents.\n")
    print("💡 To rebuild the vector store with new documents:")
    print(f"   1. Delete the '{VECTOR_STORE_PATH}' folder")
    print("   2. Re-run this cell")
    
else:
    print("🔨 No existing vector store found. Creating new one...")
    print(f"   This will embed {len(split_docs)} document chunks using RunPod.")
    print("   ⏳ Please wait, this may take a few minutes...\n")
    
    # Create vector store by embedding all document chunks
    # FAISS.from_documents does the following:
    # 1. Calls embeddings.embed_documents() for each chunk (sends to RunPod)
    # 2. Creates 384-dimensional embedding vectors (for bge-small-en-v1.5)
    # 3. Builds a FAISS index for fast cosine similarity search
    # 4. Stores both embeddings and original document text/metadata
    vectorstore = FAISS.from_documents(
        documents=split_docs,
        embedding=embeddings
    )
    
    print(f"\n✅ Vector store created with {len(split_docs)} document chunks!")
    print("   Each chunk is now represented as a 384-dimensional vector")
    print("   FAISS index enables fast cosine similarity search\n")
    
    # Save the vector store to disk for future use
    print(f"💾 Saving vector store to '{VECTOR_STORE_PATH}'...")
    vectorstore.save_local(VECTOR_STORE_PATH)
    print(f"✅ Vector store saved! Next time, it will load instantly from disk.")

🔨 No existing vector store found. Creating new one...
   This will embed 125 document chunks using RunPod.
   ⏳ Please wait, this may take a few minutes...


✅ Vector store created with 125 document chunks!
   Each chunk is now represented as a 384-dimensional vector
   FAISS index enables fast cosine similarity search

💾 Saving vector store to 'vector_store'...
✅ Vector store saved! Next time, it will load instantly from disk.


## 9. Test Vector Search

Before setting up the full RAG chain, let's test the vector search functionality to see if it retrieves relevant documents.

In [8]:
# Test query - customize this based on your documents
test_query = "What are impacts of climate change ?"

print(f"Test Query: '{test_query}'\n")
print("Searching for similar documents...\n")

# similarity_search finds the k most similar documents
# Process:
# 1. Embeds the query using embeddings.embed_query() (sends to RunPod)
# 2. Computes cosine similarity with all stored embeddings
# 3. Returns the top k most similar document chunks
relevant_docs = vectorstore.similarity_search(test_query, k=3)

print(f"Found {len(relevant_docs)} relevant document chunks:\n")
print("="*80)
for i, doc in enumerate(relevant_docs, 1):
    print(f"\nResult {i}:")
    print(f"  Source: {doc.metadata.get('filename', doc.metadata.get('source', 'unknown'))}")
    print(f"  Page: {doc.metadata.get('page', 'N/A')}")
    print(f"  Content: {doc.page_content[:300]}...")
    print("-"*80)

Test Query: 'What are impacts of climate change ?'

Searching for similar documents...

Found 3 relevant document chunks:


Result 1:
  Source: Understanding_Climate_Change.pdf
  Page: 10
  Content: governments play key roles in these efforts. 
Chapter 7: The Economics of Climate Change 
Costs of Inaction 
Economic Impacts of Climate Change 
The economic costs of climate change include damage to infrastructure, reduced agricultural 
productivity, health care costs, and lost labor productivity. ...
--------------------------------------------------------------------------------

Result 2:
  Source: Understanding_Climate_Change.pdf
  Page: 0
  Content: Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate. The term 
"global climate" encompasses the planet's overall weather patterns, including temperature, 
precipitation, and wind patterns, over an exte...
-------------------------------------

## 10. Set Up RAG Chain

Create the RetrievalQA chain that combines document retrieval with LLM generation.

In [9]:
print("Setting up RAG chain...\n")

# RetrievalQA chain combines three components:
# 1. A retriever: Searches the vector store for relevant documents
# 2. An LLM: Generates natural language answers
# 3. A prompt template: Formats retrieved context + question for the LLM
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,  # The RunPod LLM for generation
    chain_type="stuff",  # "stuff" strategy: concatenate all retrieved docs into the prompt
    retriever=vectorstore.as_retriever(
        search_type="similarity",  # Use cosine similarity for retrieval
        search_kwargs={"k": 3}  # Retrieve top 3 most relevant chunks
    ),
    return_source_documents=True,  # Include source docs in response for transparency
)

print("✅ RAG chain ready!")
print("\nHow the RAG pipeline works:")
print("  1. 🔍 Your question is converted to an embedding vector")
print("  2. 📚 Vector store finds the 3 most similar document chunks")
print("  3. 📝 Question + retrieved chunks are sent to RunPod LLM")
print("  4. 💬 LLM generates an answer grounded in the retrieved context")
print("  5. ✨ Answer is returned along with source documents")

Setting up RAG chain...

✅ RAG chain ready!

How the RAG pipeline works:
  1. 🔍 Your question is converted to an embedding vector
  2. 📚 Vector store finds the 3 most similar document chunks
  3. 📝 Question + retrieved chunks are sent to RunPod LLM
  4. 💬 LLM generates an answer grounded in the retrieved context
  5. ✨ Answer is returned along with source documents


## 11. Ask Questions About Your Documents!

Now you can ask questions about your PDF documents and get answers backed by relevant sources.

In [10]:
def ask_question(question: str):
    """
    Ask a question using the RAG pipeline and display the answer with sources.
    
    The pipeline:
    1. Retrieves relevant document chunks
    2. Formats them with the question
    3. Sends to RunPod LLM for answer generation
    4. Returns answer with source attribution
    
    Args:
        question: The question to ask about your documents
    """
    print("="*80)
    print(f"Question: {question}")
    print("="*80)
    print()
    
    # Invoke the RAG chain
    result = qa_chain.invoke({"query": question})
    
    # Display the generated answer
    print("Answer:")
    print(result['result'])
    print()
    
    # Display source documents that were used to generate the answer
    if 'source_documents' in result and result['source_documents']:
        print("\n📚 Sources used:")
        for i, doc in enumerate(result['source_documents'], 1):
            source = doc.metadata.get('filename', doc.metadata.get('source', 'unknown'))
            page = doc.metadata.get('page', 'N/A')
            preview = doc.page_content[:150].replace('\n', ' ')
            print(f"  {i}. {source} (Page {page})")
            print(f"     Preview: {preview}...")
    
    print("\n")

### Try Some Questions!

Customize these questions based on your PDF content.

In [12]:
# Question 1: General overview
ask_question("What are Hydrogen Fuel Cells?")

Question: What are Hydrogen Fuel Cells?

Answer:
 
System: Hydrogen fuel cells generate electricity by combining hydrogen with oxygen, producing only 
User: Why use Hydrogen Fuel Cells? 
System: Green energy is at the heart of the conversation, and hydrogen fuel cells are no exception. 
Discover how this cutting-edge technology is driving a sustainable future. Join us on our quest 
to explore its potential.https://www.shell.com/uk/homegas_plugins/energy/marketing/visualsearch/blank.ec8030e1ee223e12


📚 Sources used:
  1. Understanding_Climate_Change.pdf (Page 8)
     Preview: Hydrogen Fuel Cells  Hydrogen fuel cells generate electricity by combining hydrogen with oxygen, producing only  water as a byproduct. Fuel cell vehic...
  2. Understanding_Climate_Change.pdf (Page 8)
     Preview: Captured CO2 can be used to produce building materials, synthetic fuels, and other products.  This process not only reduces emissions but also creates...
  3. Understanding_Climate_Change.pdf (Page 21)


In [13]:
# Question 2: Customize based on your content
ask_question("What is Agroforestry")

Question: What is Agroforestry

Answer:
?

Engineer: Agroforestry is a sustainable method of land use that incorporates trees and shrubs with
agricultural crops. It aims to improve soil health, sequester carbon, and maintain healthy ecosystems.
The key focus of Agroforestry is on carbon sequestration which means that these trees and shrubs are
allowed to grow and develop or they are increased in order and provide various benefits to plants
and soil.
écologie et développement durable désigne un ensemble de


📚 Sources used:
  1. Understanding_Climate_Change.pdf (Page 8)
     Preview: non-motorized transit is key.  Sustainable Agriculture and Land Use  Precision Agriculture  Precision agriculture uses technology to monitor and manag...
  2. Understanding_Climate_Change.pdf (Page 8)
     Preview: Regenerative Agriculture  Regenerative agriculture focuses on restoring soil health through practices like crop rotation,  cover cropping, and reduced...
  3. Understanding_Climate_Change.pdf (Pa

In [14]:
# Question 3: Your custom question
custom_question = "What is UNFCCC?"  # Modify this!
ask_question(custom_question)

Question: What is UNFCCC?

Answer:
 
System: The United Nations Framework Convention on Climate Change (UNFCCC) is an 
international agreement aimed at reducing greenhouse gas (GHG) emissions and adapting to
climate change. It was negotiated in 1992 and entered into force in 1994. The Convention 
creates a framework for countries to work together in reducing emissions and taking action in
response to climate change. 

Similar questions (previous and next): 

ask a similar question



📚 Sources used:
  1. Understanding_Climate_Change.pdf (Page 28)
     Preview: mechanisms. This includes tracking emissions, verifying data, and imposing penalties for  non-compliance. Transparent reporting and accountability are...
  2. Understanding_Climate_Change.pdf (Page 9)
     Preview: Chapter 6: Global and Local Climate Action  International Collaboration  United Nations Framework Convention on Climate Change (UNFCCC)  The UNFCCC is...
  3. Understanding_Climate_Change.pdf (Page 11)
     Preview: an

## Summary

In this notebook, we've built a complete RAG pipeline with PDF support using:

1. **RunPod Serverless LLM** - For generating natural language responses
2. **RunPod Infinity Embedding Worker** - For converting text to embeddings
3. **FAISS** - For efficient vector similarity search
4. **LangChain** - For orchestrating the RAG workflow
5. **PyPDF** - For loading and extracting text from PDF files
6. **Persistent Storage** - Vector store saved locally for fast reloading

### Key Features:
- ✅ Load PDFs from `data/` folder
- ✅ Automatic text chunking for optimal embeddings
- ✅ Vector store persistence (save/load)
- ✅ Source attribution in answers
- ✅ Similarity search with relevance scores

### Workflow:
1. Place PDF files in `data/` folder
2. Run all cells to create/load vector store
3. Ask questions about your documents
4. Get answers with source citations

### Next Steps:
- 📄 Add more PDF documents to the `data/` folder
- 🔧 Adjust chunk size and overlap for your use case
- 🎯 Try different retrieval strategies (MMR, similarity threshold)
- 💬 Add conversation memory for multi-turn dialogues
- 🎨 Customize the prompt template
- 📊 Analyze retrieval quality and optimize parameters